<a href="https://colab.research.google.com/github/Ghan539/Gen_AI-Workshop/blob/main/Gen_AI_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import mnist


# -------------------------------------------------
# 1. LOAD MNIST DATASET
# -------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)


# -------------------------------------------------
# 2. NORMALIZE AND FLATTEN IMAGES
# -------------------------------------------------

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 28 x 28 = 784
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)


# -------------------------------------------------
# 3. PARAMETERS
# -------------------------------------------------

input_dim = 784
hidden_dim = 256
latent_dim = 2


# -------------------------------------------------
# 4. BUILD ENCODER
# -------------------------------------------------

encoder_input = layers.Input(shape=(input_dim,))

h = layers.Dense(
    hidden_dim,
    activation="relu"
)(encoder_input)

z_mean = layers.Dense(
    latent_dim,
    name="z_mean"
)(h)

z_log_var = layers.Dense(
    latent_dim,
    name="z_log_var"
)(h)


# -------------------------------------------------
# 5. SAMPLING LAYER
# -------------------------------------------------

class Sampling(layers.Layer):

    def call(self, inputs):

        z_mean, z_log_var = inputs

        epsilon = tf.random.normal(
            shape=tf.shape(z_mean)
        )

        return z_mean + tf.exp(
            0.5 * z_log_var
        ) * epsilon


z = Sampling()([z_mean, z_log_var])


# Encoder model
encoder = Model(
    encoder_input,
    [z_mean, z_log_var, z],
    name="encoder"
)


# -------------------------------------------------
# 6. BUILD DECODER
# -------------------------------------------------

latent_input = layers.Input(
    shape=(latent_dim,)
)

h_decoder = layers.Dense(
    hidden_dim,
    activation="relu"
)(latent_input)

decoder_output = layers.Dense(
    input_dim,
    activation="sigmoid"
)(h_decoder)

decoder = Model(
    latent_input,
    decoder_output,
    name="decoder"
)


# -------------------------------------------------
# 7. CREATE VAE MODEL
# -------------------------------------------------

class VAE(Model):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):

        # If data comes as (x, y), take only x
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:

            # Encoder
            z_mean, z_log_var, z = self.encoder(
                data,
                training=True
            )

            # Decoder
            reconstruction = self.decoder(
                z,
                training=True
            )

            # Reconstruction loss
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.backend.binary_crossentropy(
                        data,
                        reconstruction
                    ),
                    axis=1
                )
            )

            # KL divergence loss
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1
                    + z_log_var
                    - tf.square(z_mean)
                    - tf.exp(z_log_var),
                    axis=1
                )
            )

            # Total VAE loss
            total_loss = (
                reconstruction_loss
                + kl_loss
            )

        # Calculate gradients
        gradients = tape.gradient(
            total_loss,
            self.trainable_weights
        )

        # Update weights
        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.trainable_weights
            )
        )

        return {
            "loss": total_loss,
            "reconstruction_loss":
                reconstruction_loss,
            "kl_loss": kl_loss
        }


# -------------------------------------------------
# 8. CREATE AND COMPILE VAE
# -------------------------------------------------

vae = VAE(
    encoder,
    decoder
)

vae.compile(
    optimizer=tf.keras.optimizers.Adam()
)


# -------------------------------------------------
# 9. TRAIN VAE
# -------------------------------------------------

vae.fit(
    x_train,
    epochs=20,
    batch_size=128
)


# -------------------------------------------------
# 10. GENERATE NEW DIGITS
# -------------------------------------------------

n = 10

random_latent_vectors = np.random.normal(
    size=(n, latent_dim)
)

generated_images = decoder.predict(
    random_latent_vectors
)

plt.figure(figsize=(15, 2))

for i in range(n):

    plt.subplot(1, n, i + 1)

    plt.imshow(
        generated_images[i].reshape(28, 28),
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()


# -------------------------------------------------
# 11. VISUALIZE LATENT SPACE
# -------------------------------------------------

z_mean_test, _, _ = encoder.predict(
    x_test,
    batch_size=128
)

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    z_mean_test[:, 0],
    z_mean_test[:, 1],
    c=y_test,
    s=5
)

plt.colorbar(scatter)

plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")

plt.title("2-D Latent Space of MNIST")

plt.show()

Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)
Epoch 1/20
219/469 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - kl_loss: 13.7157 - loss: 227.8242 - reconstruction_loss: 214.1085